In [ ]:
# Kaggle settings: GPU T4 x1 | Internet ON | RAM 30GB
# Input datasets:
#   faresmamdou/farmtrust-sits             (sits_dataset.npz)
#   faresmamdou/farmtrust-sits-pretrained (sits_bert_pretrained.pt)
#   faresmamdou/farmtrust-eurocrops        (eurocrops_transfer.npz)
#   faresmamdou/farmtrust-sits-code        (sits_bert/ code)
import subprocess
import sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'torch', 'numpy', 'scikit-learn', 'hmmlearn', '-q'])

In [ ]:
import os
from pathlib import Path

import numpy as np

EURO_PATH = next(Path('/kaggle/input').rglob('eurocrops_transfer.npz'))
EGYPT_PATH = next(Path('/kaggle/input').rglob('sits_dataset_v2.npz'))
PRETRAINED_PATH = next(Path('/kaggle/input').rglob('sits_bert_pretrained.pt'))
print('EuroCrops path:', EURO_PATH)
print('Egypt path:', EGYPT_PATH)
print('Pretrained path:', PRETRAINED_PATH)
euro = np.load(EURO_PATH, allow_pickle=True)
egypt = np.load(EGYPT_PATH, allow_pickle=True)
print('EuroCrops:', euro['features'].shape)
print('Egypt:', egypt['features'].shape)

In [ ]:
import sys

def find_farmtrust_core():
    for root, dirs, files in os.walk('/kaggle/input'):
        for d in dirs:
            if d == 'sits_bert':
                path = os.path.join(root, d)
                sys.path.insert(0, os.path.dirname(path))
                return os.path.dirname(path)
    raise RuntimeError(f"sits_bert not found. Input: {os.listdir('/kaggle/input')}")

find_farmtrust_core()
from sits_bert.model import SITSBertFinetune
from sits_bert.config import FarmTrustSITSConfig
print('Model imported OK')

In [ ]:
import random
from pathlib import Path

import torch
from sklearn.metrics import f1_score, precision_score, recall_score
from torch.utils.data import DataLoader, Dataset, random_split

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
def training_device():
    if not torch.cuda.is_available():
        return torch.device('cpu')
    major, minor = torch.cuda.get_device_capability(0)
    if major < 7:
        print(f'CUDA device capability sm_{major}{minor} is not compatible with this PyTorch wheel; using CPU.')
        return torch.device('cpu')
    return torch.device('cuda')

device = training_device()
print('device:', device)

class NpzSequenceDataset(Dataset):
    def __init__(self, npz, label_key='labels'):
        self.features = npz['features'].astype(np.float32)
        self.attention_mask = npz['attention_mask'].astype(bool)
        self.doy = npz['doy'].astype(np.int64)
        labels = npz[label_key] if label_key in npz.files else npz['labels']
        self.labels = labels.astype(np.int64)
        self.stats = self._stats()

    def _stats(self):
        flat = self.features[self.attention_mask]
        mean = flat.mean(axis=0)
        std = flat.std(axis=0)
        std = np.where(std < 1e-6, 1.0, std)
        return mean.astype(np.float32), std.astype(np.float32)

    def __len__(self):
        return int(self.features.shape[0])

    def __getitem__(self, index):
        mean, std = self.stats
        x = np.nan_to_num(self.features[index], nan=0.0, posinf=0.0, neginf=0.0)
        x = np.clip((x - mean) / std, -5.0, 5.0).astype(np.float32)
        return {
            'input_ids': torch.from_numpy(x),
            'attention_mask': torch.from_numpy(self.attention_mask[index]).bool(),
            'doy': torch.from_numpy(self.doy[index]).long(),
            'labels': torch.from_numpy(self.labels[index]).long(),
        }

def load_encoder_weights(model, checkpoint_path):
    ckpt = torch.load(checkpoint_path, map_location='cpu')
    state = ckpt.get('model_state_dict', ckpt)
    encoder_state = {}
    for key, value in state.items():
        if key.startswith('encoder.'):
            encoder_state[key[len('encoder.'):]] = value
    if encoder_state:
        missing, unexpected = model.encoder.load_state_dict(encoder_state, strict=False)
    else:
        missing, unexpected = model.load_state_dict(state, strict=False)
    print('loaded pretrained weights; missing:', len(missing), 'unexpected:', len(unexpected))

def flatten_outputs(logits, labels):
    probs = torch.softmax(logits, dim=-1)
    mask = labels != -1
    return labels[mask].detach().cpu().numpy(), probs[mask].detach().cpu().numpy()

def evaluate(model, loader):
    model.eval()
    y_true_parts, y_prob_parts = [], []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = model(batch['input_ids'], batch['attention_mask'], batch['doy'])
            y_true, y_prob = flatten_outputs(logits, batch['labels'])
            y_true_parts.append(y_true)
            y_prob_parts.append(y_prob)
    y_true = np.concatenate(y_true_parts) if y_true_parts else np.asarray([], dtype=np.int64)
    y_prob = np.concatenate(y_prob_parts) if y_prob_parts else np.zeros((0, 4), dtype=np.float32)
    y_pred = y_prob.argmax(axis=1) if len(y_prob) else np.asarray([], dtype=np.int64)
    negatives = y_true != 0
    false_active = ((y_pred == 0) & negatives).sum() / max(1, negatives.sum())
    return {
        'precision_active': float(precision_score(y_true, y_pred, labels=[0], average='macro', zero_division=0)),
        'recall_active': float(recall_score(y_true, y_pred, labels=[0], average='macro', zero_division=0)),
        'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'false_active_rate': float(false_active),
    }, y_true, y_prob

def train_model(model, train_loader, val_loader, epochs, lr, class_weights, patience=None, save_path=None):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    weights = torch.tensor(class_weights, dtype=torch.float32, device=device)
    loss_fn = torch.nn.CrossEntropyLoss(weight=weights, ignore_index=-1, label_smoothing=0.1)
    history = []
    best = {'precision_active': -1.0}
    best_payload = None
    stale = 0
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        steps = 0
        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = model(batch['input_ids'], batch['attention_mask'], batch['doy'])
            loss = loss_fn(logits.reshape(-1, logits.shape[-1]), batch['labels'].reshape(-1))
            loss.backward()
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            total_loss += float(loss.detach().cpu())
            steps += 1
        avg_loss = total_loss / max(1, steps)
        metrics, y_true, y_prob = evaluate(model, val_loader)
        history.append(avg_loss)
        print(f"epoch {epoch:03d} loss={avg_loss:.5f} precision_active={metrics['precision_active']:.3f} recall_active={metrics['recall_active']:.3f} false_active_rate={metrics['false_active_rate']:.3f}")
        if metrics['precision_active'] > best['precision_active']:
            best = metrics
            best_payload = (epoch, y_true, y_prob)
            stale = 0
            if save_path:
                torch.save({'model_state_dict': model.state_dict(), 'epoch': epoch, **metrics}, save_path)
        else:
            stale += 1
            if patience is not None and stale >= patience:
                print('early stopping')
                break
    return history, best, best_payload

config = FarmTrustSITSConfig()
config.model.num_classes = 4
config.finetuning.class_weights = [2.0, 1.0, 1.2, 0.8]
model = SITSBertFinetune(config).to(device)
load_encoder_weights(model, str(PRETRAINED_PATH))

euro_dataset = NpzSequenceDataset(euro, label_key='labels')
val_size = max(1, int(0.1 * len(euro_dataset)))
train_size = len(euro_dataset) - val_size
euro_train, euro_val = random_split(euro_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(SEED))
euro_train_loader = DataLoader(euro_train, batch_size=128, shuffle=True)
euro_val_loader = DataLoader(euro_val, batch_size=128)
stage1_loss, stage1_metrics, stage1_payload = train_model(
    model,
    euro_train_loader,
    euro_val_loader,
    epochs=30,
    lr=0.0001,
    class_weights=[2.0, 1.0, 1.2, 0.8],
    save_path='/kaggle/working/sits_bert_euro_transfer.pt',
)
print('stage1 best:', stage1_metrics)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(stage1_loss)
plt.title('Stage 1 EuroCrops loss')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.grid(True)
plt.tight_layout()
plt.savefig('/kaggle/working/stage1_loss.png', dpi=150)
plt.show()

In [ ]:
config.finetuning.class_weights = [2.5, 1.0, 1.2, 0.8]
model = SITSBertFinetune(config).to(device)
transfer_ckpt = torch.load('/kaggle/working/sits_bert_euro_transfer.pt', map_location='cpu')
model.load_state_dict(transfer_ckpt['model_state_dict'], strict=False)

egypt_label_key = 'weak_labels' if 'weak_labels' in egypt.files else 'labels'
egypt_dataset = NpzSequenceDataset(egypt, label_key=egypt_label_key)
val_size = max(1, int(0.2 * len(egypt_dataset))) if len(egypt_dataset) > 1 else 1
train_size = max(1, len(egypt_dataset) - val_size)
if train_size + val_size > len(egypt_dataset):
    egypt_train = egypt_dataset
    egypt_val = egypt_dataset
else:
    egypt_train, egypt_val = random_split(egypt_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(SEED))
egypt_train_loader = DataLoader(egypt_train, batch_size=32, shuffle=True)
egypt_val_loader = DataLoader(egypt_val, batch_size=32)
stage2_loss, stage2_metrics, stage2_payload = train_model(
    model,
    egypt_train_loader,
    egypt_val_loader,
    epochs=50,
    lr=0.00005,
    class_weights=[2.5, 1.0, 1.2, 0.8],
    patience=10,
    save_path='/kaggle/working/sits_bert_transfer_finetuned.pt',
)
best_epoch, best_y_true, best_y_prob = stage2_payload
print('stage2 best:', stage2_metrics)
print('best_epoch:', best_epoch)

In [ ]:
def active_threshold_metrics(y_true, y_prob, threshold):
    y_pred = y_prob.argmax(axis=1)
    y_pred = np.where(y_prob[:, 0] >= threshold, 0, y_pred)
    precision = precision_score(y_true, y_pred, labels=[0], average='macro', zero_division=0)
    recall = recall_score(y_true, y_pred, labels=[0], average='macro', zero_division=0)
    f1 = f1_score(y_true == 0, y_pred == 0, zero_division=0)
    negatives = y_true != 0
    false_active = ((y_pred == 0) & negatives).sum() / max(1, negatives.sum())
    return float(precision), float(recall), float(f1), float(false_active)

best_threshold = 0.65
best_recall = -1.0
print('threshold | precision | recall | f1 | false_active_rate')
for threshold in np.arange(0.40, 0.91, 0.05):
    precision, recall, f1, false_active = active_threshold_metrics(best_y_true, best_y_prob, threshold)
    print(f'{threshold:.2f} | {precision:.3f} | {recall:.3f} | {f1:.3f} | {false_active:.3f}')
    if precision >= 0.80 and recall > best_recall:
        best_threshold = float(threshold)
        best_recall = float(recall)

ckpt = torch.load('/kaggle/working/sits_bert_transfer_finetuned.pt', map_location='cpu')
ckpt['optimal_threshold'] = best_threshold
ckpt['val_precision_active'] = stage2_metrics.get('precision_active')
ckpt['val_false_active_rate'] = stage2_metrics.get('false_active_rate')
torch.save(ckpt, '/kaggle/working/sits_bert_transfer_finetuned.pt')
print('optimal_threshold:', best_threshold)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

best_y_pred = best_y_prob.argmax(axis=1)
print(classification_report(best_y_true, best_y_pred, labels=[0, 1, 2, 3], target_names=['active', 'bare', 'sparse', 'uncertain'], zero_division=0))
print(confusion_matrix(best_y_true, best_y_pred, labels=[0, 1, 2, 3]))

In [ ]:
for f in os.listdir('/kaggle/working/'):
    print(f, round(os.path.getsize(f'/kaggle/working/{f}') / 1e6, 2), 'MB')

In [ ]:
print('=== TRANSFER COMPLETE. Download: sits_bert_transfer_finetuned.pt ===')